loading the enggineered data 

In [1]:
import pandas as pd
from pathlib import Path

ENGINEERED_FILE = "../data/processed/engineered_features.parquet"

df = pd.read_parquet(ENGINEERED_FILE)

print("Shape:", df.shape)
print("Number of wells:", df["WELL"].nunique())
print("Columns:", len(df.columns))

Shape: (1430974, 46)
Number of wells: 118
Columns: 46


In [2]:
wells = sorted(df["WELL"].unique())

print("Total wells:", len(wells))
print("\nFirst 20 wells:")
print(wells[:20])

print("\nLast 20 wells:")
print(wells[-20:])

Total wells: 118

First 20 wells:
['15_9-13', '15_9-14', '15_9-15', '15_9-17', '15_9-23', '16_1-2', '16_1-6 A', '16_10-1', '16_10-2', '16_10-3', '16_10-5', '16_11-1 ST3', '16_2-11 A', '16_2-16', '16_2-6', '16_2-7', '16_4-1', '16_5-3', '16_7-4', '16_7-5']

Last 20 wells:
['35_11-13', '35_11-15 S', '35_11-5', '35_11-6', '35_11-7', '35_12-1', '35_3-7 S', '35_4-1', '35_6-2 S', '35_8-4', '35_8-6 S', '35_9-10 S', '35_9-2', '35_9-5', '35_9-6 S', '35_9-7', '35_9-8', '36_7-3', '7_1-1', '7_1-2 S']


now we want the split the data as it was in the competetion not nessesory but i want so i can compare the result

In [4]:
import pandas as pd

LEADERBOARD_URL = (
    "https://raw.githubusercontent.com/"
    "bolgebrygg/Force-2020-Machine-Learning-competition/"
    "master/lithology_competition/data/leaderboard_test_features.csv"
)

HIDDEN_URL = (
    "https://raw.githubusercontent.com/"
    "bolgebrygg/Force-2020-Machine-Learning-competition/"
    "master/lithology_competition/data/hidden_test.csv"
)

leaderboard = pd.read_csv(LEADERBOARD_URL, sep=";")
hidden = pd.read_csv(HIDDEN_URL, sep=";")

print("Leaderboard shape:", leaderboard.shape)
print("Hidden shape:", hidden.shape)

print("\nLeaderboard wells:")
print(sorted(leaderboard["WELL"].dropna().unique()))

print("\nHidden wells:")
print(sorted(hidden["WELL"].dropna().unique()))

Leaderboard shape: (136786, 27)
Hidden shape: (122397, 29)

Leaderboard wells:
['15/9-14', '25/10-10', '25/11-24', '25/5-3', '29/3-1', '34/10-16 R', '34/3-3 A', '34/6-1 S', '35/6-2 S', '35/9-8']

Hidden wells:
['15/9-23', '16/2-7', '16/7-6', '17/4-1', '25/10-9', '31/2-10', '31/2-21 S', '34/3-2 S', '35/11-5', '35/9-7']


ahhh! i get the file splits

In [5]:
# Official FORCE 2020 test wells
leaderboard_wells_official = {
    "15/9-14",
    "25/10-10",
    "25/11-24",
    "25/5-3",
    "29/3-1",
    "34/10-16 R",
    "34/3-3 A",
    "34/6-1 S",
    "35/6-2 S",
    "35/9-8",
}

blind_wells_official = {
    "15/9-23",
    "16/2-7",
    "16/7-6",
    "17/4-1",
    "25/10-9",
    "31/2-10",
    "31/2-21 S",
    "34/3-2 S",
    "35/11-5",
    "35/9-7",
}

# Convert official naming convention to your LAS naming convention
def normalize_official_name(name):
    return name.replace("/", "_")

leaderboard_wells = {
    normalize_official_name(w) for w in leaderboard_wells_official
}

blind_wells = {
    normalize_official_name(w) for w in blind_wells_official
}

all_wells = set(df["WELL"].unique())

train_wells_official = all_wells - leaderboard_wells - blind_wells

print("TRAIN:", len(train_wells_official))
print("PUBLIC/LEADERBOARD:", len(leaderboard_wells))
print("BLIND TEST:", len(blind_wells))
print("TOTAL:", len(train_wells_official | leaderboard_wells | blind_wells))

print("\nPublic/Leaderboard wells:")
print(sorted(leaderboard_wells))

print("\nBlind-test wells:")
print(sorted(blind_wells))

TRAIN: 98
PUBLIC/LEADERBOARD: 10
BLIND TEST: 10
TOTAL: 118

Public/Leaderboard wells:
['15_9-14', '25_10-10', '25_11-24', '25_5-3', '29_3-1', '34_10-16 R', '34_3-3 A', '34_6-1 S', '35_6-2 S', '35_9-8']

Blind-test wells:
['15_9-23', '16_2-7', '16_7-6', '17_4-1', '25_10-9', '31_2-10', '31_2-21 S', '34_3-2 S', '35_11-5', '35_9-7']


creating split manifest


In [6]:
from pathlib import Path
import pandas as pd

split_records = []

for well in sorted(all_wells):
    if well in leaderboard_wells:
        split = "PUBLIC"
    elif well in blind_wells:
        split = "BLIND_TEST"
    elif well in train_wells_official:
        split = "TRAIN"
    else:
        raise ValueError(f"Unknown well: {well}")

    split_records.append({
        "WELL": well,
        "SPLIT": split
    })

split_df = pd.DataFrame(split_records)

# Save
split_dir = Path("../data/splits")
split_dir.mkdir(parents=True, exist_ok=True)

split_file = split_dir / "force2020_split.csv"
split_df.to_csv(split_file, index=False)

print("Saved:", split_file)
print()
print(split_df["SPLIT"].value_counts())

Saved: ../data/splits/force2020_split.csv

SPLIT
TRAIN         98
PUBLIC        10
BLIND_TEST    10
Name: count, dtype: int64


the 10 BLIND_TEST wells must not be used for model selection, feature selection, hyperparameter tuning, or threshold tuning. They are reserved for the final evaluation.

In [7]:
print(
    split_df.groupby("SPLIT")["WELL"]
    .nunique()
)

print("\nDuplicate wells:", split_df["WELL"].duplicated().sum())
print("Total wells:", split_df["WELL"].nunique())

SPLIT
BLIND_TEST    10
PUBLIC        10
TRAIN         98
Name: WELL, dtype: int64

Duplicate wells: 0
Total wells: 118


In [8]:
# Load the permanent official split
split_file = "../data/splits/force2020_split.csv"

split_df = pd.read_csv(split_file)

# Attach split information to every sample
df = df.drop(columns=["SPLIT"], errors="ignore")

df = df.merge(
    split_df,
    on="WELL",
    how="left",
    validate="many_to_one"
)

print("Shape:", df.shape)
print("\nSplit distribution:")
print(df["SPLIT"].value_counts())

print("\nWells per split:")
print(df.groupby("SPLIT")["WELL"].nunique())

Shape: (1430974, 46)

Split distribution:
SPLIT
TRAIN         1171557
PUBLIC         136841
BLIND_TEST     122576
Name: count, dtype: int64

Wells per split:
SPLIT
BLIND_TEST    10
PUBLIC        10
TRAIN         98
Name: WELL, dtype: int64


In [9]:
# Check that no well appears in more than one split
split_check = df.groupby("WELL")["SPLIT"].nunique()

assert split_check.max() == 1, "A well appears in multiple splits!"

# Sample-level datasets
train_df = df[df["SPLIT"] == "TRAIN"].copy()
public_df = df[df["SPLIT"] == "PUBLIC"].copy()
blind_df = df[df["SPLIT"] == "BLIND_TEST"].copy()

print("TRAIN:", train_df.shape)
print("PUBLIC:", public_df.shape)
print("BLIND TEST:", blind_df.shape)

print("\nWells:")
print("TRAIN:", train_df["WELL"].nunique())
print("PUBLIC:", public_df["WELL"].nunique())
print("BLIND:", blind_df["WELL"].nunique())

TRAIN: (1171557, 46)
PUBLIC: (136841, 46)
BLIND TEST: (122576, 46)

Wells:
TRAIN: 98
PUBLIC: 10
BLIND: 10


| Split | Wells | Samples |
|---|---:|---:|
| TRAIN | 98 | 1,171,557 |
| PUBLIC | 10 | 136,841 |
| BLIND_TEST | 10 | 122,576 |
| **Total** | **118** | **1,430,974** |

In [10]:
BASELINE_FEATURES = [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB"
]

TARGET = "FORCE_2020_LITHOFACIES_LITHOLOGY"

X_train_base = train_df[BASELINE_FEATURES].copy()
y_train = train_df[TARGET].copy()

X_public_base = public_df[BASELINE_FEATURES].copy()
y_public = public_df[TARGET].copy()

# Blind labels are kept separate and will NOT be used for development
X_blind_base = blind_df[BASELINE_FEATURES].copy()
y_blind = blind_df[TARGET].copy()

print("X_train:", X_train_base.shape)
print("y_train:", y_train.shape)

print("X_public:", X_public_base.shape)
print("y_public:", y_public.shape)

print("X_blind:", X_blind_base.shape)
print("y_blind:", y_blind.shape)

X_train: (1171557, 5)
y_train: (1171557,)
X_public: (136841, 5)
y_public: (136841,)
X_blind: (122576, 5)
y_blind: (122576,)


In [12]:

print("TRAIN missing:")
print(X_train_base.isna().sum())

print("\nPUBLIC missing:")
print(X_public_base.isna().sum())

print("\nBLIND missing:")
print(X_blind_base.isna().sum())

print("\nTRAIN infinite:", np.isinf(X_train_base.to_numpy()).sum())
print("PUBLIC infinite:", np.isinf(X_public_base.to_numpy()).sum())
print("BLIND infinite:", np.isinf(X_blind_base.to_numpy()).sum())

TRAIN missing:
GR              1046
RDEP_LOG10     11017
RMED_LOG10     39013
DTC            81470
RHOB          161482
dtype: int64

PUBLIC missing:
GR               55
RDEP_LOG10       59
RMED_LOG10      587
DTC             878
RHOB          16960
dtype: int64

BLIND missing:
GR             179
RDEP_LOG10      11
RMED_LOG10    9817
DTC           4222
RHOB          9542
dtype: int64

TRAIN infinite: 0
PUBLIC infinite: 0
BLIND infinite: 0


We'll load those exact training-derived values and apply them to TRAIN, PUBLIC, and BLIND. This is important because we must use the same training medians for all three sets.

In [13]:
import json

PARAMS_FILE = "../data/processed/preprocessing_params.json"

with open(PARAMS_FILE, "r") as f:
    preprocessing_params = json.load(f)

train_medians = preprocessing_params["train_medians"]

print("Training medians:")
print(train_medians)

Training medians:
{'GR': 69.434272766, 'RDEP_LOG10': 0.17010088339122542, 'RMED_LOG10': 0.16556766517280075, 'DTC': 109.1400795, 'RHOB': 2.3334783316000003}


verifying the missing values

In [14]:
for feature in BASELINE_FEATURES:
    median = train_medians[feature]

    X_train_base[feature] = X_train_base[feature].fillna(median)
    X_public_base[feature] = X_public_base[feature].fillna(median)
    X_blind_base[feature] = X_blind_base[feature].fillna(median)

print("Remaining missing values:")
print("TRAIN:", X_train_base.isna().sum().sum())
print("PUBLIC:", X_public_base.isna().sum().sum())
print("BLIND:", X_blind_base.isna().sum().sum())

print("\nRemaining infinities:")
print("TRAIN:", np.isinf(X_train_base.to_numpy()).sum())
print("PUBLIC:", np.isinf(X_public_base.to_numpy()).sum())
print("BLIND:", np.isinf(X_blind_base.to_numpy()).sum())

Remaining missing values:
TRAIN: 0
PUBLIC: 0
BLIND: 0

Remaining infinities:
TRAIN: 0
PUBLIC: 0
BLIND: 0


In [15]:
from pathlib import Path

OUTPUT_DIR = Path("../data/processed/datasets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train_base.to_parquet(OUTPUT_DIR / "X_train_baseline.parquet", index=False)
y_train.to_frame("lithology").to_parquet(
    OUTPUT_DIR / "y_train.parquet", index=False
)

X_public_base.to_parquet(OUTPUT_DIR / "X_public_baseline.parquet", index=False)
y_public.to_frame("lithology").to_parquet(
    OUTPUT_DIR / "y_public.parquet", index=False
)

X_blind_base.to_parquet(OUTPUT_DIR / "X_blind_baseline.parquet", index=False)
y_blind.to_frame("lithology").to_parquet(
    OUTPUT_DIR / "y_blind.parquet", index=False
)

print("Baseline datasets saved.")
print("Directory:", OUTPUT_DIR)

Baseline datasets saved.
Directory: ../data/processed/datasets
